# PyTorch Sampler 零基础教程

> **适用场景**：理解 DataLoader 的数据采样机制，掌握内置采样器和自定义采样器的使用方法

## 一、什么是 Sampler？

在 PyTorch 中，**Sampler（采样器）是用来决定 DataLoader 从数据集中按什么顺序取数据的工具**。

Sampler 本身**不接触数据**，它只负责生成一串**索引（index）**，告诉 DataLoader："先去拿第 5 个样本，再去拿第 2 个，再去拿第 8 个……"。

> 你可以把 Sampler 想象成一张**点餐清单**——清单上写的不是菜名，而是菜品的编号（索引）。服务员（DataLoader）就按照这个清单去仓库（Dataset）里取货。

## 二、Sampler、Dataset、DataLoader 三者的关系

用一个**餐厅运营**的类比来理解三者分工：

| 组件 | 角色 | 职责 |
|------|------|------|
| **Dataset** | 食材仓库 | 存储所有数据，提供 `__len__()`（仓库里有多少东西）和 `__getitem__(idx)`（按编号取货）两个方法 |
| **Sampler** | 点餐清单 | 生成一串数据编号（索引），决定取货的顺序 |
| **batch_sampler** | 打包员 | 把 Sampler 生成的单个索引按 `batch_size` 打包成一个个批次，比如 `[3,0,5]` 就是一个 batch 的索引 |
| **DataLoader** | 服务员 | 统筹全局——拿索引 → 去 Dataset 取数据 → 打包成 batch → 喂给模型 |

**为什么要用 Sampler？**

假设你的数据集前 500 张都是猫，后 500 张都是狗。如果 DataLoader 按顺序读取，那第一个 batch 全是猫，第二个 batch 也全是猫……这样模型根本学不到东西。Sampler 就是用来打乱这种顺序、让每个 batch 的数据更多样化的工具。

## 三、Sampler 在 DataLoader 中的使用方法

DataLoader 的构造函数中，与 Sampler 相关的参数有这几个：

```python
DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,          # 是否打乱
    sampler=None,           # 指定采样器
    batch_sampler=None,     # 指定批次采样器
    drop_last=False,        # 是否丢弃最后不足一个 batch 的数据
    ...
)
```

**关键规则**：
- 当 `shuffle=True` 时，DataLoader 会自动使用 `RandomSampler`
- 当 `shuffle=False` 时，默认使用 `SequentialSampler`
- **如果手动指定了 `sampler`，`shuffle` 参数必须设置为 `False`**
- `sampler` 和 `batch_sampler` 是**互斥**的，不能同时指定

## 四、PyTorch 内置的常用 Sampler

### 1. SequentialSampler（顺序采样）

按数据集的原始顺序（0, 1, 2, 3, ...）依次返回索引。

In [ ]:
from torch.utils.data import DataLoader, SequentialSampler, TensorDataset
import torch

# 创建一个简单的数据集
dataset = TensorDataset(torch.arange(10))
sampler = SequentialSampler(dataset)
dataloader = DataLoader(dataset, sampler=sampler, batch_size=3)

for batch in dataloader:
    print(batch)
# 输出：[0,1,2] [3,4,5] [6,7,8] [9]

### 2. RandomSampler（随机采样）

随机打乱索引顺序。如果设置 `replacement=True`，表示**有放回采样**（同一个样本可以被重复选中）。

In [ ]:
from torch.utils.data import RandomSampler

sampler = RandomSampler(dataset, replacement=False)  # 无放回，默认
dataloader = DataLoader(dataset, sampler=sampler, batch_size=3)

for batch in dataloader:
    print(batch)
# 每次运行顺序不同，例如：[7,2,5] [0,9,3] [4,1,8] [6]

### 3. SubsetRandomSampler（子集随机采样）

只从指定的索引子集中进行随机采样。常用于**划分训练集和验证集**。

In [ ]:
from torch.utils.data import SubsetRandomSampler
import numpy as np

# 假设数据集有 100 个样本
dataset_size = 100
indices = list(range(dataset_size))

# 随机打乱后按 80% / 20% 划分
np.random.shuffle(indices)
split = int(0.8 * dataset_size)
train_indices = indices[:split]
val_indices = indices[split:]

train_sampler = SubsetRandomSampler(train_indices)
val_sampler = SubsetRandomSampler(val_indices)

train_loader = DataLoader(dataset, sampler=train_sampler, batch_size=32)
val_loader = DataLoader(dataset, sampler=val_sampler, batch_size=32)

# 注意：SubsetRandomSampler 本身就带有打乱功能，不需要再设置 shuffle=True

### 4. WeightedRandomSampler（加权随机采样）

根据每个样本的**权重**来决定被选中的概率。权重越大，被选中的概率越高。**专治类别不平衡**！

**典型场景**：二分类任务中，负类 950 个，正类只有 50 个。如果直接训练，模型会把所有样本都预测为负类，准确率还能高达 95%，但这个模型毫无意义。

In [ ]:
from torch.utils.data import WeightedRandomSampler
import torch

# 假设标签：0 有 900 个，1 有 100 个
labels = [0] * 900 + [1] * 100

# 计算每个类别的权重：样本少的类别权重高
class_counts = [900, 100]
class_weights = [1.0 / count for count in class_counts]  # [0.0011, 0.01]

# 为每个样本分配权重
sample_weights = [class_weights[label] for label in labels]

# 创建采样器，replacement=True 表示有放回采样
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)

# 这样，原本占比 10% 的正类样本被选中的概率大大提升，模型就不会"偏科"了

### 5. BatchSampler（批次采样器）

`BatchSampler` 本身不直接产生单个索引，而是把其他 Sampler 产生的索引**按 batch_size 打包**成一个个批次。

In [ ]:
from torch.utils.data import BatchSampler, SequentialSampler

# 先用 SequentialSampler 生成 [0,1,2,3,4,5,6,7]
base_sampler = SequentialSampler(dataset)
# 打包成 batch_size=3 的批次：[[0,1,2], [3,4,5], [6,7]]
batch_sampler = BatchSampler(base_sampler, batch_size=3, drop_last=False)

dataloader = DataLoader(dataset, batch_sampler=batch_sampler)

# 注意：使用 batch_sampler 时，不能再指定 batch_size、shuffle 和 sampler

## 五、如何自定义 Sampler

如果内置的 Sampler 满足不了你的需求，可以自己写一个。只需要**继承 `Sampler` 类**，并实现两个方法：

- `__iter__()`：返回一个**索引迭代器**，决定采样顺序
- `__len__()`：返回**总样本数**

In [ ]:
from torch.utils.data import Sampler

class EvenFirstSampler(Sampler):
    """自定义采样器：先取偶数索引，再取奇数索引"""
    
    def __init__(self, data_source):
        self.data_source = data_source

    def __iter__(self):
        # 先取所有偶数索引，再取所有奇数索引
        indices = list(range(len(self.data_source)))
        even = [i for i in indices if i % 2 == 0]
        odd = [i for i in indices if i % 2 == 1]
        return iter(even + odd)

    def __len__(self):
        return len(self.data_source)

# 使用自定义采样器
sampler = EvenFirstSampler(dataset)
dataloader = DataLoader(dataset, sampler=sampler, batch_size=3)
# 输出顺序：[0,2,4] [6,8,1] [3,5,7] [9]

## 六、实际应用场景总结

| 场景 | 推荐 Sampler |
|------|-------------|
| 普通训练，需要打乱数据 | `RandomSampler`（或直接 `shuffle=True`） |
| 测试/验证，需要按顺序评估 | `SequentialSampler`（或直接 `shuffle=False`） |
| 划分训练集和验证集 | `SubsetRandomSampler` |
| 类别严重不平衡 | `WeightedRandomSampler` |
| 多卡分布式训练 | `DistributedSampler` |
| 需要自定义取数逻辑 | 自定义 Sampler |

## 七、常见问题与注意事项

### 1. sampler 和 shuffle 不能同时使用

```python
# ❌ 错误写法
DataLoader(dataset, sampler=my_sampler, shuffle=True)

# ✅ 正确写法
DataLoader(dataset, sampler=my_sampler, shuffle=False)
```

### 2. batch_sampler 与 sampler、batch_size 互斥

```python
# ❌ 错误写法
DataLoader(dataset, batch_sampler=my_batch_sampler, batch_size=32)

# ✅ 正确写法
DataLoader(dataset, batch_sampler=my_batch_sampler)
```

### 3. 分布式训练记得调用 set_epoch()

使用 `DistributedSampler` 时，每个 epoch 开始前需要调用 `sampler.set_epoch(epoch)`，确保每个 epoch 的数据划分是随机且不重叠的。

### 4. Sampler 只适用于 Map-style Dataset

如果你用的是 `IterableDataset`（迭代式数据集），数据加载顺序由数据集本身的 `__iter__()` 控制，Sampler 不生效。

## 八、总结

**一句话记住 Sampler**：

> Sampler 就是一张"取货清单"，它只告诉 DataLoader 按什么顺序去 Dataset 里拿数据，自己从来不碰数据本身。

掌握 Sampler，你就能灵活控制数据的加载顺序——无论是随机打乱、划分数据集、处理类别不平衡，还是实现自定义的采样策略，都能轻松应对。

## 九、附录：GRPO 中的分布式 K 重复采样器

在 GRPO（Group Relative Policy Optimization）算法中，需要一个特殊的采样器来实现**组内优势计算**：同一个 prompt 生成 k 个图像，在同一 batch 中比较奖励。

`DistributedKRepeatSampler` 的设计要点：
- 每轮抽取 m 个不同的 prompt
- 每个 prompt 重复 k 次（生成 k 个变体）
- 将重复后的样本分配给各 GPU
- 通过 all_gather 重新聚合奖励，实现全局组内统计

In [ ]:
from torch.utils.data import Sampler
import torch

class DistributedKRepeatSampler(Sampler):
    """每轮抽取 m 个 prompt，每个 prompt 重复 k 次，再切给各 GPU。
    
    total_samples = world_size * batch_size，m = total_samples/k
    同一 prompt 的 k 个图像样本可在一个全局 batch 中比较奖励，
    正是组内 advantage 的来源。
    
    Args:
        dataset: 数据集（包含 prompt 列表）
        batch_size: 每个 GPU 的 batch 大小
        k: 每个 prompt 的重复次数（GRPO group size）
        num_replicas: GPU 数量（分布式进程数）
        rank: 当前 GPU 的编号
        seed: 随机种子
    """
    
    def __init__(self, dataset, batch_size, k, num_replicas, rank, seed=0):
        self.dataset = dataset
        self.batch_size = batch_size
        self.k = k
        self.num_replicas = num_replicas
        self.rank = rank
        self.seed = seed

        self.total_samples = self.num_replicas * self.batch_size
        
        # 确保总数能被 k 整除
        assert (
            self.total_samples % self.k == 0
        ), f"k can not div n*b, k{k}-num_replicas{num_replicas}-batch_size{batch_size}"
        
        # m = 需要抽取的 prompt 数量
        self.m = self.total_samples // self.k
        self.epoch = 0

    def __iter__(self):
        """无限迭代器，每轮产生一批索引"""
        while True:
            g = torch.Generator()
            g.manual_seed(self.seed + self.epoch)
            
            # 第一步：无放回抽取 m 个不同 prompt
            indices = torch.randperm(len(self.dataset), generator=g)[: self.m].tolist()
            
            # 第二步：把每个索引连续复制 k 次
            # 例如 indices=[0,5,2], k=4 → repeated_indices=[0,0,0,0, 5,5,5,5, 2,2,2,2]
            repeated_indices = [idx for idx in indices for _ in range(self.k)]

            # 第三步：打乱所有重复样本
            # 这样同一 prompt 的 k 份可能分散在不同 GPU
            shuffled_indices = torch.randperm(
                len(repeated_indices), generator=g
            ).tolist()
            shuffled_samples = [repeated_indices[i] for i in shuffled_indices]

            # 第四步：按 GPU 切片分配
            per_card_samples = []
            for i in range(self.num_replicas):
                start = i * self.batch_size
                end = start + self.batch_size
                per_card_samples.append(shuffled_samples[start:end])
            
            # 返回当前 GPU 的样本索引
            yield per_card_samples[self.rank]

    def __len__(self):
        """返回总样本数（每个 epoch 的样本数）"""
        return self.total_samples

    def set_epoch(self, epoch):
        """设置 epoch，确保每个 epoch 的随机种子不同"""
        self.epoch = epoch

### 使用示例

In [ ]:
# 假设场景：4 张 GPU，每张卡 batch_size=8，每个 prompt 重复 4 次
num_gpus = 4
batch_size_per_gpu = 8
k_repeat = 4

# 计算：总样本数 = 4 * 8 = 32，需要抽取的 prompt 数 = 32 / 4 = 8
# 即每轮抽 8 个不同 prompt，每个 prompt 重复 4 次，共 32 个样本

sampler = DistributedKRepeatSampler(
    dataset=my_dataset,
    batch_size=batch_size_per_gpu,
    k=k_repeat,
    num_replicas=num_gpus,
    rank=0,  # 当前 GPU 编号
    seed=42
)

dataloader = DataLoader(my_dataset, batch_sampler=sampler)

# 每个 epoch 开始前调用
for epoch in range(num_epochs):
    sampler.set_epoch(epoch)
    for batch in dataloader:
        # 训练代码
        pass

### 核心设计思想

**为什么需要 "先重复再打乱"？**

1. **保证多样性**：每轮先无放回抽取 m 个不同 prompt，确保组间多样性
2. **组内比较**：每个 prompt 重复 k 次，为 GRPO 的组内优势计算提供基础
3. **负载均衡**：打乱后按 GPU 切片，确保每张卡的负载相同
4. **全局聚合**：虽然同一 prompt 的 k 份可能分散在不同 GPU，但通过 `all_gather` 可以重新聚合奖励，实现全局组内统计

这种设计使得 GRPO 算法能够在分布式环境下高效地计算**per-prompt advantage**，而无需额外的 critic 网络。